In [5]:
#Importing necessary libraries
import cv2
import numpy as np
from matplotlib import pyplot as plt

#Node Structure/class
l=0
R=1
class Node:
   def __init__(self, key=None,freq=None):
      self.freq = freq
      self.key = key
      self.left = None
      self.right = None

#1.Read it to memory from the file

img = cv2.imread("./humans1.jpg")


rows,cols,dims = img.shape

# img_box_filtered = np.zeros((rows,cols,dims),np.uint8)

pixels_dict=dict()

#2.Making dictionary for the pixel-values and frequency of that pixel-value

for i in range(rows):
    for j in range(cols):
        for k in range(dims):
            if img[i,j,k] in pixels_dict:
                pixels_dict[img[i,j,k]]+=1
            else:
                pixels_dict[img[i,j,k]]=1

# print(len(pixels_dict))

min_heap=[0]

#3.Add all values into min-heap

for key in pixels_dict.keys():
    newNode = Node(key,pixels_dict[key])
    min_heap.append(newNode)


#min-heap

def PARENT(i):
 return int(i/2)

def LEFT(i):
 return 2*i

def RIGHT(i):
 return 2*i + 1

#inser to min-heap
def insertToMinHeap(A,x):
  A.append(x)
  i = len(A)-1
  while(i>1 and A[PARENT(i)].freq > A[i].freq):
    temp = A[PARENT(i)]
    A[PARENT(i)]= A[i]
    A[i]=temp
    i=PARENT(i)


#min-heapify
def minHeapify(A,i):
    L=LEFT(i)
    r=RIGHT(i)

    if L < len(A) and A[L].freq < A[i].freq:
      smallest = L
    else:
      smallest = i
    
    if r < len(A) and A[r].freq < A[smallest].freq:
      smallest = r
    
    if(smallest != i):
      temp = A[smallest]
      A[smallest]=A[i]
      A[i]=temp
      minHeapify(A,smallest)

#min-heap minimum
def minHeapMinimum(A):
  if (len(A)>0):
    return A[1]
  else:
    return -1

#min-heap extract min
def minHeapExtractMin(A):
  min = minHeapMinimum(A)
  min_heap[1]=min_heap[-1]
  min_heap.pop()
  minHeapify(A,1)
  return min

#build min-heap
def buildMinHeap(A,n):
  n=int(n/2)
  for i in range(0,n):
    minHeapify(A,n-i)

# Huffman Coding for Image Compression

# Step 1: Build the min-heap
buildMinHeap(min_heap,len(min_heap))

# Step 2: Build the Huffman Tree
while(len(min_heap)>2):
    x = minHeapExtractMin(min_heap)
    y = minHeapExtractMin(min_heap)
    z = Node(None,x.freq+y.freq)
    z.left = x
    z.right = y
    insertToMinHeap(min_heap,z)

# Step 3: Assign binary codes to each leaf node

hammingCodes = {}

def assignCodes(node, code):
    if node.key is not None:
        # Leaf node
        hammingCodes[node.key] = code
    else:
        # Internal node
        assignCodes(node.left, code + '0')
        assignCodes(node.right, code + '1')

assignCodes(min_heap[1], '')

# Step 4: Compress the image using the Huffman codes
compressed_image = ''
for i in range(rows):
    for j in range(cols):
        for k in range(dims):
            compressed_image += hammingCodes[img[i,j,k]]

def pad_encoded_image(encoded_image):
        extra_padding = 8 - len(encoded_image) % 8
        for i in range(extra_padding):
            encoded_image += "0"
        padded_info = "{0:08b}".format(extra_padding)
        encoded_image = padded_info + encoded_image
        return encoded_image
encoded_image = pad_encoded_image(compressed_image)
def get_byte_array(padded_encoded_image):
    if(len(padded_encoded_image) % 8 != 0):
        print("Encoded image not padded properly")
        exit(0)
    b = bytearray()
    for i in range(0, len(padded_encoded_image), 8):
        byte = padded_encoded_image[i:i+8]
        b.append(int(byte, 2))
    return b

byte_array = get_byte_array(encoded_image)

with open("humans1_compressed.bin", 'wb') as output:
    output.write(byte_array)

# # Step 5: Store the Huffman codes and the compressed image in a file
# with open('huffman_codes.txt', 'w') as file:
#     for key, value in hammingCodes.items():
#         file.write(str(key) + ',' + value + '\n')
# with open('compressed_image.txt', 'w') as file:
#     file.write(compressed_image)

# # Step 6: Read the Huffman codes and the compressed image from the file
# codes = {}
# with open('huffman_codes.txt', 'r') as file:
#     for line in file:
#         key, value = line.strip().split(',')
#         codes[int(key)] = value
# compressed_image = ''
# with open('compressed_image.txt', 'r') as file:
#     compressed_image = file.read()

# # Step 7: Decompress the image using the Huffman codes
# decoded_image = np.zeros((rows,cols,dims), np.uint8)
# code = ''
# k = 0
# for i in range(rows):
#     for j in range(cols):
#         for p in range(dims):
#             while(1):
#               code += compressed_image[k]
#               if code in codes.values():
#                   index = list(codes.values()).index(code)
#                   decoded_image[i,j,p] = list(codes.keys())[index]
#                   code = ''
#                   k +=1
#                   break
#               k += 1
# # Step 8: Display the original and decoded images
# plt.subplot(1, 2, 1)
# plt.title('Original Image')
# plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
# plt.subplot(1, 2, 2)
# plt.title('Decoded Image')
# plt.imshow(cv2.cvtColor(decoded_image, cv2.COLOR_BGR2RGB))
# plt.show()

# # Step 9: Write the decoded image to a file
# cv2.imwrite('decoded_image.jpg', decoded_image)


In [8]:

import re
import numpy as np
from PIL import Image
print("Huffman Compression Program")
print("=================================================================")
h = int(input("Enter 1 if you want to input an colour image file, 2 for default gray scale case:"))
if h == 1:
    file = input("Enter the filename:")
    my_string = np.asarray(Image.open(file),np.uint8)
    shape = my_string.shape
    a = my_string
    print ("Enetered string is:",my_string)
    my_string = str(my_string.tolist())
elif h == 2:
    array = np.arange(0, 737280, 1, np.uint8)
    my_string = np.reshape(array, (1024, 720))
    print ("Enetered string is:",my_string)
    a = my_string
    my_string = str(my_string.tolist())

else:
    print("You entered invalid input")                    # taking user input

letters = []
only_letters = []
for letter in my_string:
    if letter not in letters:
        frequency = my_string.count(letter)             #frequency of each letter repetition
        letters.append(frequency)
        letters.append(letter)
        only_letters.append(letter)

nodes = []
while len(letters) > 0:
    nodes.append(letters[0:2])
    letters = letters[2:]                               # sorting according to frequency
nodes.sort()
huffman_tree = []
huffman_tree.append(nodes)                             #Make each unique character as a leaf node

def combine_nodes(nodes):
    pos = 0
    newnode = []
    if len(nodes) > 1:
        nodes.sort()
        nodes[pos].append("1")                       # assigning values 1 and 0
        nodes[pos+1].append("0")
        combined_node1 = (nodes[pos] [0] + nodes[pos+1] [0])
        combined_node2 = (nodes[pos] [1] + nodes[pos+1] [1])  # combining the nodes to generate pathways
        newnode.append(combined_node1)
        newnode.append(combined_node2)
        newnodes=[]
        newnodes.append(newnode)
        newnodes = newnodes + nodes[2:]
        nodes = newnodes
        huffman_tree.append(nodes)
        combine_nodes(nodes)
    return huffman_tree                                     # huffman tree generation

newnodes = combine_nodes(nodes)

huffman_tree.sort(reverse = True)
print("Huffman tree with merged pathways:")

checklist = []
for level in huffman_tree:
    for node in level:
        if node not in checklist:
            checklist.append(node)
        else:
            level.remove(node)
count = 0
for level in huffman_tree:
    print("Level", count,":",level)             #print huffman tree
    count+=1
print()

letter_binary = []
if len(only_letters) == 1:
    lettercode = [only_letters[0], "0"]
    letter_binary.append(letter_code*len(my_string))
else:
    for letter in only_letters:
        code =""
        for node in checklist:
            if len (node)>2 and letter in node[1]:           #genrating binary code
                code = code + node[2]
        lettercode =[letter,code]
        letter_binary.append(lettercode)
print(letter_binary)
print("Binary code generated:")
for letter in letter_binary:
    print(letter[0], letter[1])

bitstring =""
for character in my_string:
    for item in letter_binary:
        if character in item:
            bitstring = bitstring + item[1]
binary ="0b"+bitstring
print("Your message as binary is:")
                                        # binary code generated

uncompressed_file_size = len(my_string)*7
compressed_file_size = len(binary)-2
print("Your original file size was", uncompressed_file_size,"bits. The compressed size is:",compressed_file_size)
print("This is a saving of ",uncompressed_file_size-compressed_file_size,"bits")
output = open("compressed.txt","w+")
print("Compressed file generated as compressed.txt")
output = open("compressed.txt","w+")
print("Decoding.......")
output.write(bitstring)

bitstring = str(binary[2:])
uncompressed_string =""
code =""
for digit in bitstring:
    code = code+digit
    pos=0                                        #iterating and decoding
    for letter in letter_binary:
        if code ==letter[1]:
            uncompressed_string=uncompressed_string+letter_binary[pos] [0]
            code=""
        pos+=1

print("Your UNCOMPRESSED data is:")
if h == 1:
    temp = re.findall(r'\d+', uncompressed_string)
    res = list(map(int, temp))
    res = np.array(res)
    res = res.astype(np.uint8)
    res = np.reshape(res, shape)
    print(res)
    print("Observe the shapes and input and output arrays are matching or not")
    print("Input image dimensions:",shape)
    print("Output image dimensions:",res.shape)
    data = Image.fromarray(res)
    data.save('uncompressed.png')
    if a.all() == res.all():
        print("Success")
if h == 2:
    temp = re.findall(r'\d+', uncompressed_string)
    res = list(map(int, temp))
    print(res)
    res = np.array(res)
    res = res.astype(np.uint8)
    res = np.reshape(res, (1024, 720))
    print(res)
    data = Image.fromarray(res)
    data.save('uncompressed.png')
    print("Success")

Huffman Compression Program


ValueError: invalid literal for int() with base 10: ''